# EDA: Track A Dataset Analysis
This notebook provides an empirical justification for the Path Ranking Algorithm's engineering choices. The analysis uses an out-of-core approach to prevent memory overflow, particularly for 2-hop and 3-hop datasets.

In [1]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
BASE_DIR = Path.cwd().resolve().parent / "pra_pipeline" / "dataset_processed"

## 1. Dimensionality, Sparsity, and Class Balance
The function below extracts three key metrics in a single pass:
1. **Dimensionality**: Number of features extracted directly from the Parquet schema.
2. **Sparsity**: The percentage of `0.0` values in the feature matrix.
3. **Class Balance**: The ratio between correct answers (`1`) and negative samples (`0`).

In [6]:
def get_stats(hop_num):
    file_path = BASE_DIR / f"{hop_num}-hop" / "train_set.parquet"
    
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return
        
    pq_file = pq.ParquetFile(file_path)
    
    feature_cols = [c for c in pq_file.schema.names if c not in ['question_id', 'answer', 'label']]
    
    print(f"--- Stats for {hop_num}-hop Train Set ---")
    print(f"Total features: {len(feature_cols)}")
    
    total_cells = 0
    zero_cells = 0
    labels_1 = 0
    labels_0 = 0
    
    for batch in pq_file.iter_batches(batch_size=50000):
        df = batch.to_pandas()
        
        labels_1 += (df['label'] == 1).sum()
        labels_0 += (df['label'] == 0).sum()
        
        X = df[feature_cols].values
        total_cells += X.size
        zero_cells += np.count_nonzero(X == 0.0) 
        
    sparsity = (zero_cells / total_cells) * 100 if total_cells > 0 else 0
    
    print(f"Class Balance: {labels_1} (Positives) / {labels_0} (Negatives)")
    print(f"Negative to Positive Ratio: {labels_0 / labels_1:.2f}:1")
    print(f"Matrix Sparsity: {sparsity:.2f}%\n")

In [7]:
get_stats(1)

--- Stats for 1-hop Train Set ---
Total features: 36
Class Balance: 184799 (Positives) / 521002 (Negatives)
Negative to Positive Ratio: 2.82:1
Matrix Sparsity: 96.38%



In [8]:
get_stats(2)

--- Stats for 2-hop Train Set ---
Total features: 396
Class Balance: 717091 (Positives) / 366130 (Negatives)
Negative to Positive Ratio: 0.51:1
Matrix Sparsity: 99.48%



In [9]:
get_stats(3)

--- Stats for 3-hop Train Set ---
Total features: 4244
Class Balance: 1353164 (Positives) / 512713 (Negatives)
Negative to Positive Ratio: 0.38:1
Matrix Sparsity: 95.41%



## 2. Top Feature Relevance
By calculating the mean probability of each feature across the dataset, we extract the top 5 most activated paths. This confirms that the Random Walk successfully prioritizes semantically relevant connections. Duplicate interaction features (`_X_q`) are filtered out for clarity.

In [13]:
def get_top_features(hop_num, top_k=5):
    file_path = BASE_DIR / f"{hop_num}-hop" / "train_set.parquet"
    
    if not file_path.exists():
        print(f"File not found: {file_path}")
        return
        
    pq_file = pq.ParquetFile(file_path)
    
    
    base_cols = [c for c in pq_file.schema.names 
                 if c not in ['question_id', 'answer', 'label'] and not c.endswith('_X_q')]
    
    feature_sums = pd.Series(0.0, index=base_cols)
    total_rows = 0
    
    for batch in pq_file.iter_batches(batch_size=50000):
        df = batch.to_pandas()
        total_rows += len(df)
        feature_sums += df[base_cols].sum()
        
    feature_means = feature_sums / total_rows
    top_features = feature_means.sort_values(ascending=False).head(top_k)
    
    print(f"--- Top {top_k} Features for {hop_num}-hop Train Set ---")
    for idx, (feat, val) in enumerate(top_features.items(), 1):
        print(f"{idx}. {feat:<40} | Mean Prob: {val:.4f}")
    print("\n")

In [14]:
get_top_features(1)

--- Top 5 Features for 1-hop Train Set ---
1. release_year                             | Mean Prob: 0.0969
2. directed_by                              | Mean Prob: 0.0916
3. starred_actors                           | Mean Prob: 0.0820
4. written_by                               | Mean Prob: 0.0812
5. has_genre                                | Mean Prob: 0.0777




In [15]:
get_top_features(2)

--- Top 5 Features for 2-hop Train Set ---
1. inv_starred_actors_THEN_starred_actors   | Mean Prob: 0.0345
2. inv_written_by_THEN_written_by           | Mean Prob: 0.0325
3. inv_directed_by_THEN_directed_by         | Mean Prob: 0.0184
4. starred_actors_THEN_inv_starred_actors   | Mean Prob: 0.0087
5. directed_by_THEN_inv_directed_by         | Mean Prob: 0.0084




In [16]:
get_top_features(3)

--- Top 5 Features for 3-hop Train Set ---
1. directed_by_THEN_inv_directed_by_THEN_directed_by | Mean Prob: 0.0136
2. written_by_THEN_inv_written_by_THEN_written_by | Mean Prob: 0.0080
3. has_genre_THEN_inv_has_genre_THEN_has_genre | Mean Prob: 0.0071
4. written_by_THEN_inv_written_by_THEN_directed_by | Mean Prob: 0.0051
5. starred_actors_THEN_inv_starred_actors_THEN_starred_actors | Mean Prob: 0.0048




## Dimensionality, Sparsity, and Class Balance

| Hop Level | Total Features | Class Balance (Pos / Neg) | Neg:Pos Ratio | Matrix Sparsity |
| :--- | :--- | :--- | :--- | :--- |
| **1-hop** | 36 | 184,799 / 521,002 | 2.82 : 1 | 96.38% |
| **2-hop** | 396 | 717,091 / 366,130 | 0.51 : 1 | 99.48% |
| **3-hop** | 4,244 | 1,353,164 / 512,713 | 0.38 : 1 | 95.41% |

* **Combinatorial Explosion:** The feature space expands exponentially from 36 columns at 1-hop to 4,244 columns at 3-hop. This scaling empirically validates the necessity of the out-of-core processing architecture and L1 regularization to maintain tractability.
* **Class Distribution Shift:** The 1-hop dataset is dominated by negative samples (2.82:1). However, at 2-hop and 3-hop, the balance flips to favor positive instances (0.51:1 and 0.38:1, respectively). This proves the negative sampling strategy functioned correctly, capping the exponentially growing false candidates to prevent dataset saturation.
* **Persistent Sparsity:** The matrix remains highly sparse across all iterations, peaking at 99.48% for 2-hop. The slight sparsity reduction in 3-hop (95.41%) indicates that while the feature space is massive, the random walks heavily converge on a few densely connected hub entities.

---

## Top Feature Relevance Analysis

The path extraction pipeline successfully prioritizes semantically meaningful graph traversals over random noise:

* **1-Hop (Direct Fact Retrieval):** The dominant paths (`release_year`, `directed_by`, `starred_actors`, `written_by`, `has_genre`) strictly mirror the primary relations required to answer single-step questions.
* **2-Hop (Cyclic Co-occurrence):** The model heavily relies on inverse-then-forward relationship chains (e.g., `inv_starred_actors_THEN_starred_actors` at 0.0345 mean probability, `inv_written_by_THEN_written_by` at 0.0325). These paths logically represent "co-worker" or "shared characteristic" structures, bridging an actor to a co-star via a shared movie.
* **3-Hop (Complex Transitivity):** The highest-scoring paths utilize deep transitive logic, such as `directed_by_THEN_inv_directed_by_THEN_directed_by` (0.0136). This pattern illustrates the model navigating from an entity to a director, moving to another movie by that director, and returning to the director—a robust topological pattern for reinforcing entity disambiguation over long distances.